# <font color="green"> **AI for Cybersecurity Project 2024-25 Group 9**

<font color="black">
Lamb Giovanni - 0622702168 <br>
Orlando Palma - 0622702433 <br>
Saturnino Fabrizio - 0622702328 <br>
Zottarelli Egidio - 0622702175

## **Contents**
1. Definitions
2. Preprocessing of the dataset
3. Evaluation

In [ ]:
%%capture
!pip install git+https://github.com/rcmalli/keras-vggface.git
!pip install numpy==1.23.5
!pip install tensorflow==2.12
!pip install keras==2.12
!pip install keras_vggface
!pip install keras_applications
!pip install mtcnn

## **Import**

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image

from keras_vggface.vggface import VGGFace
from keras_vggface.utils import preprocess_input

**Alternative Classifier Testing**
In this section, our goal is to broaden the evaluation of facial recognition classifiers by incorporating an additional model trained on the VGG-Face2 dataset. This new classifier, referred to as NN2, will serve as a counterpart to our existing NN1 model in the comparative analysis. For this purpose, we adopt **ResNet-50**, a deep convolutional neural network widely recognized for its strong performance in image recognition tasks. ResNet-50 is particularly effective in learning discriminative features, making it well-suited for face recognition scenarios such as those posed by the VGG-Face2 dataset.

## **1. Definitions**

Here are defined:
- the function to process the images so that they can be given as input to ResNet50
- The function to evaluate the accuracy of the ResNeto50 network.

In [ ]:
def preprocess_face(face, required_size=(224, 224)):
    """
    Preprocess a single face image for input to ResNet50.

    Parameters:
    - face: numpy array, expected shape (3, 160, 160), the input face image.
    - required_size: tuple, default (224, 224), the target size for the output image.

    Returns:
    - face_array: numpy array, shape (224, 224, 3), preprocessed face image.
    """
    # Resize the face to the required size
    # plt.imshow(np.transpose(face, (1,2,0)))
    if face.shape != (3, 160, 160):
        raise ValueError(f"Unexpected face shape: {face.shape}, expected (3, 160, 160)")

    # Ensure the image is in uint8 format
    if face.dtype != np.uint8:
        face = ((face + 1.0) * 127.5).astype('uint8')

    # Convert (3, 160, 160) to (160, 160, 3)
    face = face.transpose(1, 2, 0)

    # Check if the image is already in the correct shape and dtype
    if face.shape == (224, 224, 3) and face.dtype == np.uint8:
        return face

    # Resize the face to the required size
    face_image = Image.fromarray(face)
    face_image = face_image.resize(required_size)
    face_array = np.asarray(face_image)

    # print(face_array)
    # plt.imshow(np.transpose(face_array, (0,1,2)))

    return np.asarray(face_image).astype('float32')

def preprocess(images):
    """
    Preprocess a batch of images for input to ResNet50.

    Parameters:
    - images: list or array of numpy arrays, each representing a face image.

    Returns:
    - images: numpy array, shape (N, 224, 224, 3), preprocessed images ready for model input.
    """
    # Preprocess each image
    images = [preprocess_face(img) for img in images]

    # Stack into a single array: shape (N, 224, 224, 3)
    images = np.array(images)

    # Apply VGGFace preprocessing (ResNet50 = version 2)
    return preprocess_input(images, version=2)

In [ ]:
def get_accuracy_keras(model, x_test, y_test):
    """
    Evaluate the accuracy of a Keras model on test data.

    Parameters:
    - model: Trained Keras model to use for prediction.
    - x_test: Numpy array, test images or input data for the model.
    - y_test: Numpy array or list, true labels for the test data.

    Returns:
    - accuracy: Float, the accuracy of the model on the test set.
    - correct_predictions: Integer, number of correctly classified samples.
    - total_predictions: Integer, total number of samples evaluated.
    """
    predictions = model.predict(x_test, batch_size=32)

    predicted_classes = np.argmax(predictions, axis=1)
    true_values = np.array(y_test)

    results_df = pd.DataFrame({
        'Prediction': predicted_classes,
        'True Value': true_values
    })

    results_df.to_csv("prediction_nn2_resnet50.csv", index=False)

    correct_predictions = np.sum(predicted_classes == true_values)
    total_predictions = len(true_values)

    accuracy = correct_predictions / total_predictions if total_predictions > 0 else 0

    return accuracy, correct_predictions, total_predictions


## **2. Preprocessing of the dataset**

Obtaining a dataset containing images suitable for ResNet50

In [ ]:
# Loading the model
nn2 = VGGFace(model='resnet50')

In [ ]:
# Loading the data

x_test = np.load('datsets/x_test_32.npy')
y_test = np.load('datasets/y_test_32.npy')

In [ ]:
# new dataset

x_test_resnet50 = preprocess(x_test)

In [ ]:
np.save('x_test_resnet50.npy', x_test_resnet50)

In [ ]:
x_test_resnet50 = np.load('datasets/x_test_resnet50.npy')
y_test = np.load('datasets/y_test_32.npy')

In [ ]:
print("Shape:", x_test_resnet50.shape)
print("Dtype:", x_test_resnet50.dtype)
print("Min value:", x_test_resnet50.min())
print("Max value:", x_test_resnet50.max())


## **3. Evaluation**

Accuracy evaluation of ResNet50


In [ ]:
accuracy, correct_predictions, total_predictions = get_accuracy_keras(nn2, x_test_resnet50, y_test)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Correctly classified: {correct_predictions}")
print(f"Incorrectly classified: {total_predictions - correct_predictions}")

32/32 [==============================] - 78s 2s/step
Accuracy: 94.40%
Correctly classified: 944
Incorrectly classified: 56
